In [41]:
import pandas as pd
import numpy as np
import ast
import datetime
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error,r2_score,accuracy_score,classification_report,confusion_matrix

In [42]:
df = pd.read_csv("movie.csv")

In [43]:
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [44]:
# removing spaces " "
df.columns = df.columns.str.strip()

In [45]:
df = df[(df['budget']>0) & (df['revenue']>0)]

In [46]:
df.isnull().sum()

budget                     0
genres                     0
homepage                1882
id                         0
keywords                   0
original_language          0
original_title             0
overview                   0
popularity                 0
production_companies       0
production_countries       0
release_date               0
revenue                    0
runtime                    0
spoken_languages           0
status                     0
tagline                  245
title                      0
vote_average               0
vote_count                 0
dtype: int64

In [47]:
df=df.drop(['tagline','homepage'],axis=1)

In [48]:
df.columns

Index(['budget', 'genres', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'title', 'vote_average', 'vote_count'],
      dtype='object')

In [49]:
df.isnull().sum()

budget                  0
genres                  0
id                      0
keywords                0
original_language       0
original_title          0
overview                0
popularity              0
production_companies    0
production_countries    0
release_date            0
revenue                 0
runtime                 0
spoken_languages        0
status                  0
title                   0
vote_average            0
vote_count              0
dtype: int64

In [50]:
print((df['runtime']==0).any())

False


In [51]:
if 'release_date' in df.columns:
    df['release_date']=pd.to_datetime(df['release_date'],errors='coerce')
    df=df.dropna(subset=['release_date'])
    df['release_year']=df['release_date'].dt.year
    df['release_month']=df['release_date'].dt.month
    

In [52]:
# Convert all big value in short value like revenue is 1100000 and 4000 so model will miss learn it think which value is big it's most preferanciable
df['log_budget']=np.log1p(df['budget']) 
df['log_revenue']=np.log1p(df['revenue']) 


In [53]:
def extract_generes(x):
    try:
        genres = ast.literal.eval() if isinstance(x, str) else []
        return len(genres)
    except:
        return 0

if 'geners' in df.columns:
    df['genres_count'] = df['genres'].apply(extract_genres)
else:
    df['genre_count'] = 0  # fallback if 'genres' column doesn't exist

In [54]:
df.columns

Index(['budget', 'genres', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'title', 'vote_average', 'vote_count',
       'release_year', 'release_month', 'log_budget', 'log_revenue',
       'genre_count'],
      dtype='object')

In [55]:
drop_columns = ['id','title','orignal_data','status','vote_average', 'vote_count']
df = df.drop(columns=[c for c in drop_columns if c in df.columns])

In [56]:

df['hit'] = (df['revenue'] > df['budget']).astype(int)


In [57]:
# Normalize revenue by median of release year
median_by_year = df.groupby('release_year')['revenue'].median()
df['adj_revenue'] = df.apply(lambda x: x['revenue'] / median_by_year[x['release_year']], axis=1)

In [58]:
# 1️ Keep top 5 production countries
if 'production_countries' in df.columns:
    top_countries = df['production_countries'].value_counts().head(5).index
    df['top_country'] = df['production_countries'].apply(lambda x: x if x in top_countries else 'Other')

In [59]:
print("Preprocessing complete. Dataset shape:", df.shape)

Preprocessing complete. Dataset shape: (3229, 21)


In [60]:
df.columns

Index(['budget', 'genres', 'keywords', 'original_language', 'original_title',
       'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'release_year', 'release_month', 'log_budget',
       'log_revenue', 'genre_count', 'hit', 'adj_revenue', 'top_country'],
      dtype='object')

In [61]:
df.isnull().sum()

budget                  0
genres                  0
keywords                0
original_language       0
original_title          0
overview                0
popularity              0
production_companies    0
production_countries    0
release_date            0
revenue                 0
runtime                 0
spoken_languages        0
release_year            0
release_month           0
log_budget              0
log_revenue             0
genre_count             0
hit                     0
adj_revenue             0
top_country             0
dtype: int64

In [62]:
# Correct feature names
features = ['log_budget', 'runtime', 'popularity', 'genre_count', 'adj_revenue']

if 'top_country' in df.columns:
    features.append('top_country')

# Select features
X = df[features]

# One-hot encode categorical columns
X = pd.get_dummies(X, columns=['top_country'], drop_first=True)

# Target
y = df['hit']


In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [71]:

model = RandomForestClassifier(
    n_estimators=100,      # number of trees
    max_depth=10,          # limit depth to reduce memorization
    random_state=42
)


In [72]:
model.fit(X_train,y_train)

RandomForestClassifier(max_depth=10, random_state=42)

In [73]:
# Predict on test set
y_pred = model.predict(X_test)

In [74]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.955108359133127

Confusion Matrix:
 [[140  18]
 [ 11 477]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.89      0.91       158
           1       0.96      0.98      0.97       488

    accuracy                           0.96       646
   macro avg       0.95      0.93      0.94       646
weighted avg       0.95      0.96      0.95       646



In [75]:
# On training data
y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"✅ Training Accuracy: {train_accuracy*100:.2f}%")

# On test data
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"✅ Test Accuracy: {test_accuracy*100:.2f}%")


✅ Training Accuracy: 99.34%
✅ Test Accuracy: 95.51%
